## 1. 로드: 팀원 위키 페이지 로드

In [1]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# 팀 저장소의 members/<이름>/wiki/pages/*.md 를 전부 읽는다
WIKI_ROOT = Path(r"C:\Users\chltp\Documents\NLP\Study-service_basic\members")

loader = DirectoryLoader(
    str(WIKI_ROOT),
    glob="*/wiki/pages/*.md",                  # 모든 팀원의 위키 페이지
    loader_cls=TextLoader,                      # .md를 그냥 글자로 읽기
    loader_kwargs={"encoding": "utf-8"},        # 한글이 깨지지 않게
)
docs = loader.load()

# 출처로 쓸 짧은 이름 붙이기: "mingi/redis-ttl-cache"
for d in docs:
    p = Path(d.metadata["source"])
    d.metadata["page"] = f"{p.parts[-4]}/{p.stem}"

print("문서 수:", len(docs))
for d in docs:
    print(f"{d.metadata['page']:<40} {len(d.page_content):>6}자")


C:\Users\chltp\AppData\Local\Temp\ipykernel_32960\1955805366.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


문서 수: 16
addie/async-when-needed                    1014자
doyu/async-fastapi                         1421자
huiji/image-vs-container                    425자
huiji/when-to-use-async                     455자
mingi/api-key-env                          1749자
mingi/async-fastapi                        1876자
mingi/container-commands                   1631자
mingi/docker-compose                       1953자
mingi/dockerfile-instructions              1991자
mingi/fastapi-basics                       1466자
mingi/image-container-layer-registry       1056자
mingi/redis-ttl-cache                      1887자
mingi/vm-conda-docker                       615자
mingi/volume-port                          1104자
sejin/1-docker-concepts                   10122자
sejin/2-fastapi-redis-async                9542자


## 2. 분할: 500자 청크로 자르고 ID 붙이기

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 500        # 세션에서 200/500/1000을 바꿔 볼 수 있게 변수로 둔다
CHUNK_OVERLAP = 50      # 앞뒤 청크가 50자씩 겹치게

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)     # 문서 16개 → 청크 여러 개

# 청크마다 고유 ID 붙이기: "페이지#번호"  (예: mingi/redis-ttl-cache#2)
count = {}
for c in chunks:
    page = c.metadata["page"]
    count[page] = count.get(page, 0) + 1
    c.metadata["id"] = f"{page}#{count[page]}"

print("청크 수:", len(chunks))
for page, n in count.items():
    print(f"{page:<40} {n:>3}개")

print("\n--- 첫 청크 ---")
print(chunks[0].metadata)
print(chunks[0].page_content)


청크 수: 112
addie/async-when-needed                    3개
doyu/async-fastapi                         4개
huiji/image-vs-container                   1개
huiji/when-to-use-async                    1개
mingi/api-key-env                          5개
mingi/async-fastapi                        5개
mingi/container-commands                   5개
mingi/docker-compose                       7개
mingi/dockerfile-instructions              7개
mingi/fastapi-basics                       5개
mingi/image-container-layer-registry       4개
mingi/redis-ttl-cache                      6개
mingi/vm-conda-docker                      2개
mingi/volume-port                          5개
sejin/1-docker-concepts                   27개
sejin/2-fastapi-redis-async               25개

--- 첫 청크 ---
{'source': 'C:\\Users\\chltp\\Documents\\NLP\\Study-service_basic\\members\\addie\\wiki\\pages\\async-when-needed.md', 'page': 'addie/async-when-needed', 'id': 'addie/async-when-needed#1'}
# async 함수는 언제 필요한가 (초안)

## 한 줄 요약

<!-- TODO: 직접 

## 3 임베딩 : bge-m3

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",                        # 내 PC에 받아 둔 모델 (API 비용 0원)
    encode_kwargs={"normalize_embeddings": True},    # 벡터 길이를 1로 맞춤 (아래 설명)
)

v = embeddings.embed_query("Redis TTL이 뭐야?")
print("벡터 길이:", len(v))
print("앞 5개 숫자:", [round(x, 4) for x in v[:5]])
print("벡터 크기(길이):", round(sum(x * x for x in v) ** 0.5, 4))


c:\Users\chltp\Documents\NLP\tool스터디\week3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 30519.64it/s]


벡터 길이: 1024
앞 5개 숫자: [-0.0877, 0.0199, -0.0229, 0.0507, 0.0247]
벡터 크기(길이): 1.0


### 실험: 비슷한 뜻의 문장이 유사도가 더 높은가? (1에 더 가까운가?)

- 기준 문장 `q`: "Redis TTL이 뭐야?"
- 비교 문장 `s1`: "캐시에 저장한 값의 유효기간은 어떻게 정해?" → 뜻이 비슷함 (단어는 다름)
- 비교 문장 `s2`: "도커 이미지 크기를 줄이는 방법" → 상관없음

**예상:** `s1`이 `s2`보다 `q`와 뜻이 비슷하므로, `q`와의 유사도가 더 커야 한다.

**방법:** 세 문장을 bge-m3로 임베딩한 뒤 내적을 계산한다. 벡터 길이를 1로 맞췄으므로(`normalize_embeddings=True`) 내적 = 코사인 유사도이고, 1에 가까울수록 비슷하다.

**확인할 점:** `s1`에는 "TTL"이라는 단어가 없다. 그래도 유사도가 높게 나오면, 임베딩은 단어가 아니라 **뜻**으로 비교한다는 뜻이다.


In [5]:
q  = embeddings.embed_query("Redis TTL이 뭐야?")
s1 = embeddings.embed_query("캐시에 저장한 값의 유효기간은 어떻게 정해?")   # 뜻이 비슷함 (단어는 다름)
s2 = embeddings.embed_query("도커 이미지 크기를 줄이는 방법")             # 상관없음

dot = lambda a, b: sum(x * y for x, y in zip(a, b))   # 길이가 1이라 내적 = 코사인 유사도
print("TTL ↔ 캐시 유효기간:", round(dot(q, s1), 3))
print("TTL ↔ 도커 이미지  :", round(dot(q, s2), 3))


TTL ↔ 캐시 유효기간: 0.358
TTL ↔ 도커 이미지  : 0.27


**결과:** TTL ↔ 캐시 유효기간 = 0.358, TTL ↔ 도커 이미지 = 0.27 → 예상대로 (s1 > s2)

- 단어("TTL")가 겹치지 않아도 뜻이 비슷한 문장이 더 높게 나옴 → 임베딩은 뜻으로 비교한다
- 상관없는 문장도 0이 아니라 0.27이 나옴 → 절댓값보다 "어느 쪽이 더 높은가(순서)"가 중요. 검색도 점수가 가장 높은 k개를 고른다


## 4. 저장 : Chroma에 112개 넣기

In [10]:
import time
from langchain_chroma import Chroma

t = time.perf_counter()
vectordb = Chroma.from_documents(
    documents=chunks,                                  # 112개 청크
    embedding=embeddings,                              # 각 청크를 bge-m3로 벡터화
    ids=[c.metadata["id"] for c in chunks],            # 청크 ID를 DB의 ID로도 사용
    collection_name=f"team_wiki_{CHUNK_SIZE}",         # 청크 크기별로 따로 저장
    persist_directory="chroma_db",                     # week3/chroma_db 폴더에 저장
    collection_metadata={"hnsw:space": "cosine"},      # 코사인 거리로 가까운 것 찾기
)
print(f"저장된 청크: {vectordb._collection.count()}개, 걸린 시간: {time.perf_counter() - t:.1f}초")


저장된 청크: 112개, 걸린 시간: 109.1초


In [11]:
item = vectordb.get(ids=["mingi/redis-ttl-cache#2"], include=["documents", "metadatas", "embeddings"])
print("ID      :", item["ids"][0])
print("metadata:", item["metadatas"][0])
print("원문    :", item["documents"][0][:150], "...")
print("벡터    :", len(item["embeddings"][0]), "개 숫자, 앞 5개", [round(x, 4) for x in item["embeddings"][0][:5]])


ID      : mingi/redis-ttl-cache#2
metadata: {'page': 'mingi/redis-ttl-cache', 'id': 'mingi/redis-ttl-cache#2', 'source': 'C:\\Users\\chltp\\Documents\\NLP\\Study-service_basic\\members\\mingi\\wiki\\pages\\redis-ttl-cache.md'}
원문    : ```json
{
  "answer": "안녕하세요! 반갑습니다. 오늘 어떤 도움이 필요하신가요?",
  "cached": true,
  "elapsed_ms": 0.9623000005376525
}
```

이미 저장된 질문들을 저장하기 위한 시스템,캐시 히트일경우  ...
벡터    : 1024 개 숫자, 앞 5개 [np.float64(-0.0374), np.float64(-0.0031), np.float64(-0.0174), np.float64(0.0545), np.float64(-0.0096)]


## 5. 검색: retriever로 비슷한 청크 찾기

### 5-1. 점수와 함께 검색

In [12]:
questions = [
    "Redis TTL이 뭐야?",
    "도커 이미지와 컨테이너의 차이는?",
    "async는 언제 써야 해?",
]

for q in questions:
    print(f"\n질문: {q}")
    for doc, score in vectordb.similarity_search_with_score(q, k=5):
        print(f"  {score:.3f}  {doc.metadata['id']}")



질문: Redis TTL이 뭐야?
  0.368  mingi/redis-ttl-cache#1
  0.376  mingi/redis-ttl-cache#2
  0.409  sejin/2-fastapi-redis-async#8
  0.454  mingi/redis-ttl-cache#4
  0.477  sejin/2-fastapi-redis-async#9

질문: 도커 이미지와 컨테이너의 차이는?
  0.241  huiji/image-vs-container#1
  0.325  mingi/image-container-layer-registry#2
  0.337  mingi/image-container-layer-registry#1
  0.340  sejin/1-docker-concepts#1
  0.342  mingi/container-commands#4

질문: async는 언제 써야 해?
  0.284  huiji/when-to-use-async#1
  0.306  doyu/async-fastapi#1
  0.308  sejin/2-fastapi-redis-async#19
  0.310  addie/async-when-needed#1
  0.317  mingi/async-fastapi#1


### 5-2. retriever 만들기

In [13]:
retriever = vectordb.as_retriever(search_kwargs={"k": 5})

docs = retriever.invoke("Redis TTL이 뭐야?")
print(len(docs), "개")
for d in docs:
    print(" ", d.metadata["id"])


5 개
  mingi/redis-ttl-cache#1
  mingi/redis-ttl-cache#2
  sejin/2-fastapi-redis-async#8
  mingi/redis-ttl-cache#4
  sejin/2-fastapi-redis-async#9


## 6. 생성: LCEL 체인으로 답 + 출처

### 6-1. .env 키 불러오기 + Claude 준비

In [14]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()                                            # week3/.env 를 환경 변수로 읽기
print("키 있음:", bool(os.getenv("ANTHROPIC_API_KEY")))   # 키 값은 출력하지 않고 있는지만 확인

llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=16000)   # 2주차와 같은 모델

resp = llm.invoke("RAG를 한 문장으로 설명해 줘.")
print(resp.text)
print(resp.usage_metadata)                               # 토큰 수 (2주차처럼 비용 계산에 씀)


키 있음: True
RAG(검색 증강 생성)는 언어 모델이 답변을 만들기 전에 외부 문서에서 관련 정보를 검색해 함께 참고하게 함으로써, 더 정확하고 최신의 답변을 생성하는 기법입니다.
{'input_tokens': 27, 'output_tokens': 94, 'total_tokens': 121, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_token_details': {'reasoning': 0}}


### 6-2. LCEL 체인 만들기

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 프롬프트: 자료만 근거로, 없으면 없다고
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 AI 엔지니어링 스터디의 튜터다. 아래 [자료]만 근거로 한국어로 3~5문장으로 답한다. "
               "자료에 없는 내용이면 '위키에 없는 내용입니다'라고 답한다.\n\n[자료]\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    """청크 5개를 [페이지 이름] + 내용 형태의 글 하나로 합친다"""
    return "\n\n".join(f"[{d.metadata['page']}]\n{d.page_content}" for d in docs)

answer_chain = prompt | llm | StrOutputParser()          # 프롬프트 → Claude → 글자만

rag_chain = (
    {"docs": retriever, "question": RunnablePassthrough()}           # ① 검색 + 질문 그대로
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # ② 청크를 글로
    | RunnablePassthrough.assign(answer=answer_chain)                 # ③ 답 만들기
)

def ask(q):
    out = rag_chain.invoke(q)
    sources = list(dict.fromkeys(d.metadata["page"] for d in out["docs"]))   # 출처 페이지 (중복 제거)
    return out["answer"], sources

answer, sources = ask("Redis TTL은 몇 초로 정했어?")
print(answer)
print("출처:", sources)


자료에 따르면 `mingi/redis-ttl-cache` 프로젝트의 TTL은 **3600초(1시간)**로 설정되어 있고, 1시간이 지나면 저장된 값이 자동으로 삭제됩니다. TTL이 너무 짧으면 캐시 효과가 줄고, 너무 길면 바뀐 내용이 반영되지 않은 채 오래 남을 수 있다고 정리되어 있습니다. `sejin/2-fastapi-redis-async` 문서의 `EX 60`(60초)은 설명을 위한 예시 값이므로 프로젝트 설정으로 보지 않는 것이 맞습니다.
출처: ['mingi/redis-ttl-cache', 'sejin/2-fastapi-redis-async']


### 6-3. 질문 5개: 답과 출처 기록

In [16]:
import time

questions = [
    "Redis TTL은 몇 초로 정했어?",
    "도커 이미지와 컨테이너의 차이는?",
    "async는 언제 써야 해?",
    "docker compose에서 api 컨테이너는 redis에 어떤 주소로 접속해?",
    "쿠버네티스 파드가 뭐야?",          # 위키에 없는 질문
]

records = []
for q in questions:
    t0 = time.perf_counter()
    answer, sources = ask(q)
    sec = time.perf_counter() - t0
    records.append({"question": q, "answer": answer, "sources": sources, "sec": round(sec, 2)})
    print(f"\n■ {q}  ({sec:.2f}초)\n{answer}\n출처: {sources}")



■ Redis TTL은 몇 초로 정했어?  (3.62초)
mingi/redis-ttl-cache 자료에 따르면 TTL은 **3600초(1시간)**로 설정했고, 1시간이 지나면 값이 자동으로 삭제됩니다. 자료에서는 TTL이 너무 짧으면 캐시 효과가 줄고, 너무 길면 값이 바뀌어도 오래된 값이 남는다고 설명합니다. sejin/2-fastapi-redis-async의 `EX 60`은 문법을 보여 주기 위한 예시 값이라 이 프로젝트의 설정값이 아닙니다.
출처: ['mingi/redis-ttl-cache', 'sejin/2-fastapi-redis-async']

■ 도커 이미지와 컨테이너의 차이는?  (2.94초)
이미지는 컨테이너를 실행하는 데 필요한 파일, 라이브러리, 코드, 설정을 묶은 **읽기 전용 완성품**이고, 컨테이너는 그 이미지를 실행한 **격리된 실행 환경**입니다. 비유하면 이미지는 도면이나 밀키트이고, 컨테이너는 그것으로 만든 결과물입니다. 컨테이너는 이미지 위에 얇은 쓰기 레이어를 올려 실행되므로, 컨테이너 안에서 바꾼 내용은 원본 이미지를 변경하지 않습니다. 하나의 이미지로 여러 컨테이너를 띄울 수 있다는 점도 둘의 관계에서 중요합니다.
출처: ['huiji/image-vs-container', 'mingi/image-container-layer-registry', 'sejin/1-docker-concepts', 'mingi/container-commands']

■ async는 언제 써야 해?  (2.43초)
`async def`는 LLM 호출, Redis, DB, 외부 API처럼 응답을 기다리는 시간이 대부분인 I/O 작업에 쓰면 좋습니다. 기다리는 동안 서버가 다른 요청을 받을 수 있기 때문입니다. CPU를 계속 쓰는 계산(모델 추론, 큰 계산)은 async로 빨라지지 않으니 `def`로 두면 FastAPI가 별도 스레드에서 실행합니다. `async def` 안에서는 `redis.asyncio`, `httpx` 같은 async 라이브러리

In [17]:
print(llm.invoke("Redis TTL은 몇 초로 정했어?").text)

이 대화에서는 Redis TTL을 정한 적이 없어서, 어떤 설정을 말씀하시는지 알 수 없어요. 이전 대화나 코드, 설정 파일이 여기 없는 것 같습니다.

TTL 값을 확인하려면 아래 방법을 써 보세요.

- **redis-cli로 특정 키 확인**: `TTL 키이름`을 실행하면 남은 시간(초)이 나오고, `-1`은 만료 시간이 없다는 뜻, `-2`는 키가 없다는 뜻입니다.
- **코드에서 확인**: `SET`이나 `EXPIRE`를 호출하는 부분에서 초 단위 값을 찾아보시면 됩니다.
- **설정 파일 확인**: 캐시 라이브러리(Spring Cache, Django cache 등)의 설정에 기본 TTL이 들어 있는 경우가 많습니다.

코드나 설정을 붙여 주시면 TTL 값과 적절성을 함께 검토해 드릴게요.


## 7. 평가: Recall@5, MRR@5, 응답 시간

### 7-1. 질문셋 불러오기

In [ ]:
all_pages = sorted({c.metadata["page"] for c in chunks})
print(len(all_pages)); print(*all_pages, sep="\n")


16
addie/async-when-needed
doyu/async-fastapi
huiji/image-vs-container
huiji/when-to-use-async
mingi/api-key-env
mingi/async-fastapi
mingi/container-commands
mingi/docker-compose
mingi/dockerfile-instructions
mingi/fastapi-basics
mingi/image-container-layer-registry
mingi/redis-ttl-cache
mingi/vm-conda-docker
mingi/volume-port
sejin/1-docker-concepts
sejin/2-fastapi-redis-async


In [32]:
eval_set = [
    {"question": "캐시 유효기간을 정하는 방법은?",              "answer_pages": ["mingi/redis-ttl-cache", "sejin/2-fastapi-redis-async"]},
    {"question": "이미지와 컨테이너는 어떻게 달라?",            "answer_pages": ["huiji/image-vs-container", "mingi/image-container-layer-registry", "sejin/1-docker-concepts"]},
    {"question": "도커 레이어 캐시는 어떻게 동작해?",           "answer_pages": ["mingi/image-container-layer-registry", "sejin/1-docker-concepts"]},
    {"question": "실행 중인 컨테이너 목록을 보거나 멈추는 명령은?", "answer_pages": ["mingi/container-commands", "sejin/1-docker-concepts"]},
    {"question": "컨테이너를 꺼도 파일이 남게 하려면?",          "answer_pages": ["mingi/volume-port", "sejin/1-docker-concepts"]},
    {"question": "가상머신, 콘다, 도커는 뭐가 달라?",            "answer_pages": ["mingi/vm-conda-docker"]},
    {"question": "서버와 Redis를 한 번에 띄우는 방법은?",        "answer_pages": ["mingi/docker-compose", "sejin/2-fastapi-redis-async"]},
    {"question": "FastAPI에서 await를 빼먹으면 어떻게 돼?",     "answer_pages": ["huiji/when-to-use-async", "doyu/async-fastapi", "addie/async-when-needed", "mingi/async-fastapi", "sejin/2-fastapi-redis-async"]},
    {"question": "API 비밀 키를 코드에 직접 안 쓰고 관리하려면?",       "answer_pages": ["mingi/api-key-env", "sejin/2-fastapi-redis-async"]},
    {"question": "이미지를 만들 때 설계도 파일에 쓰는 명령어는 뭐가 있어?", "answer_pages": ["mingi/dockerfile-instructions", "sejin/1-docker-concepts"]},
    {"question": "파이썬으로 웹 API 서버를 처음 만들려면 어떻게 해?",     "answer_pages": ["mingi/fastapi-basics", "sejin/2-fastapi-redis-async"]},
]


print(len(eval_set), "개")


11 개


### 7-2. 평가 함수: Recall@5, MRR@5

In [33]:
def evaluate(eval_set):
    hits, rr_sum, rows = 0, 0.0, []
    for item in eval_set:
        pages = [d.metadata["page"] for d in retriever.invoke(item["question"])]   # 청크 5개의 페이지 (순위순)
        rank = next((i + 1 for i, p in enumerate(pages) if p in item["answer_pages"]), None)  # 정답이 처음 나온 순위
        hits += rank is not None
        rr_sum += 1 / rank if rank else 0
        rows.append((rank, item["question"], pages[0]))
    n = len(eval_set)
    return hits / n, rr_sum / n, rows


### 7-3. 점수 측정

In [34]:
recall, mrr, rows = evaluate(eval_set)
print(f"Recall@5 = {recall:.3f},  MRR@5 = {mrr:.3f}  (질문 {len(rows)}개, CHUNK_SIZE={CHUNK_SIZE})")
for rank, q, top in rows:
    print(f"  {rank if rank else '✗':>2}  {top:<35}  {q}")

Recall@5 = 1.000,  MRR@5 = 1.000  (질문 11개, CHUNK_SIZE=500)
   1  mingi/redis-ttl-cache                캐시 유효기간을 정하는 방법은?
   1  huiji/image-vs-container             이미지와 컨테이너는 어떻게 달라?
   1  sejin/1-docker-concepts              도커 레이어 캐시는 어떻게 동작해?
   1  mingi/container-commands             실행 중인 컨테이너 목록을 보거나 멈추는 명령은?
   1  sejin/1-docker-concepts              컨테이너를 꺼도 파일이 남게 하려면?
   1  mingi/vm-conda-docker                가상머신, 콘다, 도커는 뭐가 달라?
   1  sejin/2-fastapi-redis-async          서버와 Redis를 한 번에 띄우는 방법은?
   1  sejin/2-fastapi-redis-async          FastAPI에서 await를 빼먹으면 어떻게 돼?
   1  mingi/api-key-env                    API 비밀 키를 코드에 직접 안 쓰고 관리하려면?
   1  mingi/dockerfile-instructions        이미지를 만들 때 설계도 파일에 쓰는 명령어는 뭐가 있어?
   1  sejin/2-fastapi-redis-async          파이썬으로 웹 API 서버를 처음 만들려면 어떻게 해?
